# Out-of-workbench training with a Kubeflow `PyTorchJob`

Alternative to running [../04-train-yolo.ipynb](../04-train-yolo.ipynb) or the [../pipelines/](../pipelines/) DAG. Here the workbench just *submits* a single Kubeflow `PyTorchJob` Custom Resource; the actual training runs in a dedicated pod that the RHOAI Training Operator schedules on the cluster.

The pitch for a customer: **one yaml, fire-and-forget, GPU-requestable**. You can close the workbench, go get coffee, and the cluster keeps training. When the pod finishes, `best.pt` (plus `args.yaml` + `results.csv`) is on S3 at the same prefix the Elyra pipeline writes to — so [../05-register-and-deploy.ipynb](../05-register-and-deploy.ipynb) can register and deploy it with no changes.

**Prerequisites on the cluster:**
- `pytorchjobs.kubeflow.org` CRD installed (part of RHOAI's Distributed Workloads).
- `aws-connection-storage` secret with the usual `AWS_*` keys.
- A `workshop-dataset` prefix already populated on S3 by pipeline node 01, OR run the Elyra node once to seed it.

In [ ]:
%%capture
!pip install kubernetes boto3==1.35.55 botocore==1.35.55

In [ ]:
import os
import time
from pathlib import Path

from kubernetes import client, config

JOB_NAME        = os.environ.get("PYTORCHJOB_NAME", "ppe-yolo-train")
NAMESPACE       = os.environ.get("KUBERNETES_NAMESPACE", "ppe-detection-cv")
SERVICE_ACCOUNT = os.environ.get("SERVICE_ACCOUNT", "aws-connection-storage")

# Ultralytics ship an official image with torch + yolo pre-installed; we just
# pip-install boto3/pyyaml on container start. Swap to `latest` (not -cpu) if
# your cluster has a GPU node you want to target.
IMAGE = os.environ.get("TRAIN_IMAGE", "ultralytics/ultralytics:latest-cpu")

# Hyperparameters (match 04-train-yolo.ipynb and pipelines/02-train.ipynb).
EPOCHS   = int(os.environ.get("EPOCHS", "100"))
IMG_SIZE = int(os.environ.get("IMG_SIZE", "640"))
BATCH    = int(os.environ.get("BATCH", "16"))
LR0      = float(os.environ.get("LR0", "0.001"))
FREEZE   = int(os.environ.get("FREEZE", "10"))
PATIENCE = int(os.environ.get("PATIENCE", "20"))

# S3 layout (same contract as pipelines/02-train.ipynb).
DATASET_PREFIX = os.environ.get("DATASET_PREFIX", "workshop-dataset")
MODEL_PREFIX   = os.environ.get("MODEL_PREFIX",   "models/ppe-yolov8n-vlm")
MODEL_VERSION  = os.environ.get("MODEL_VERSION",  "v1")

print(f"PyTorchJob {NAMESPACE}/{JOB_NAME}")
print(f"  image: {IMAGE}")
print(f"  hyperparams: epochs={EPOCHS}, imgsz={IMG_SIZE}, batch={BATCH}, lr0={LR0}, freeze={FREEZE}, patience={PATIENCE}")
print(f"  dataset s3 prefix: {DATASET_PREFIX}")
print(f"  model s3 prefix:   {MODEL_PREFIX}/{MODEL_VERSION}")

## The script the pod will run

Same logic as [`../pipelines/02-train.ipynb`](../pipelines/02-train.ipynb) — download dataset from S3, fine-tune YOLOv8n, upload `best.pt` + provenance back to S3. Embedded inline as a string so this notebook stays self-contained (no ConfigMap to manage).

In [ ]:
TRAINING_SCRIPT = r'''
import os, shutil, sys
from pathlib import Path
import boto3, botocore
from ultralytics import YOLO

BUCKET         = os.environ["AWS_S3_BUCKET"]
DATASET_PREFIX = os.environ["DATASET_PREFIX"].rstrip("/")
MODEL_PREFIX   = os.environ["MODEL_PREFIX"].rstrip("/")
MODEL_VERSION  = os.environ["MODEL_VERSION"]
EPOCHS   = int(os.environ.get("EPOCHS", "100"))
IMG_SIZE = int(os.environ.get("IMG_SIZE", "640"))
BATCH    = int(os.environ.get("BATCH", "16"))
LR0      = float(os.environ.get("LR0", "0.001"))
FREEZE   = int(os.environ.get("FREEZE", "10"))
PATIENCE = int(os.environ.get("PATIENCE", "20"))

STAGE = Path("/tmp/ppe-train")
DATA_DIR = STAGE / "data"
RUNS_DIR = STAGE / "runs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3 = session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
bucket = s3.Bucket(BUCKET)

# 1. Download dataset from S3
count = 0
for obj in bucket.objects.filter(Prefix=f"{DATASET_PREFIX}/"):
    rel = obj.key[len(DATASET_PREFIX) + 1:]
    if not rel:
        continue
    dst = DATA_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    bucket.download_file(obj.key, str(dst))
    count += 1
print(f"[job] downloaded {count} objects from s3://{BUCKET}/{DATASET_PREFIX}", flush=True)

# 2. Fine-tune
model = YOLO("yolov8n.pt")
model.train(
    data=str(DATA_DIR / "dataset.yaml"),
    epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
    lr0=LR0, freeze=FREEZE, patience=PATIENCE,
    project=str(RUNS_DIR), name="ppe-vlm", exist_ok=True,
)
SAVE_DIR = Path(model.trainer.save_dir)
best_pt = SAVE_DIR / "weights" / "best.pt"
assert best_pt.is_file(), f"training did not produce {best_pt}"

# 3. Upload best.pt + provenance
s3_prefix = f"{MODEL_PREFIX}/{MODEL_VERSION}"
bucket.upload_file(str(best_pt), f"{s3_prefix}/best.pt")
bucket.upload_file(str(DATA_DIR / "dataset.yaml"), f"{s3_prefix}/data.yaml")
for extra in ("args.yaml", "results.csv"):
    src = SAVE_DIR / extra
    if src.is_file():
        bucket.upload_file(str(src), f"{s3_prefix}/{extra}")

print(f"[job] uploaded weights + provenance to s3://{BUCKET}/{s3_prefix}/", flush=True)
'''

# Wrap in a bash command that pip-installs missing deps, then runs the script.
SHELL_WRAPPER = (
    "set -e\n"
    "pip install --quiet boto3==1.35.55 pyyaml 1>&2\n"
    "python - <<'PYEOF'\n"
    f"{TRAINING_SCRIPT}\n"
    "PYEOF\n"
)
print(f"Script size: {len(SHELL_WRAPPER)} chars")

## Build and submit the PyTorchJob

Single-node training (one `Worker`, replicas=1) — YOLO doesn't use distributed training for our tiny 40-image dataset anyway. The point here is the *envelope*: this same pod could request a GPU or scale to multi-node with two line changes.

In [ ]:
pytorchjob = {
    "apiVersion": "kubeflow.org/v1",
    "kind": "PyTorchJob",
    "metadata": {
        "name": JOB_NAME,
        "namespace": NAMESPACE,
        "labels": {"workshop": "rhelai-cv-ppe-demo"},
    },
    "spec": {
        "runPolicy": {"cleanPodPolicy": "None"},  # keep pod around so we can read logs
        "pytorchReplicaSpecs": {
            "Master": {
                "replicas": 1,
                "restartPolicy": "OnFailure",
                "template": {
                    "metadata": {"labels": {"workshop": "rhelai-cv-ppe-demo"}},
                    "spec": {
                        "serviceAccountName": SERVICE_ACCOUNT,
                        "containers": [{
                            "name": "pytorch",
                            "image": IMAGE,
                            "command": ["bash", "-c", SHELL_WRAPPER],
                            "env": [
                                {"name": "DATASET_PREFIX", "value": DATASET_PREFIX},
                                {"name": "MODEL_PREFIX",   "value": MODEL_PREFIX},
                                {"name": "MODEL_VERSION",  "value": MODEL_VERSION},
                                {"name": "EPOCHS",   "value": str(EPOCHS)},
                                {"name": "IMG_SIZE", "value": str(IMG_SIZE)},
                                {"name": "BATCH",    "value": str(BATCH)},
                                {"name": "LR0",      "value": str(LR0)},
                                {"name": "FREEZE",   "value": str(FREEZE)},
                                {"name": "PATIENCE", "value": str(PATIENCE)},
                            ],
                            # Pull all AWS_* keys from the data-connection secret.
                            "envFrom": [{"secretRef": {"name": "aws-connection-storage"}}],
                            "resources": {
                                "requests": {"cpu": "1",  "memory": "4Gi"},
                                "limits":   {"cpu": "4",  "memory": "8Gi"},
                                # Uncomment on a GPU cluster:
                                # "limits": {"cpu": "4", "memory": "16Gi", "nvidia.com/gpu": "1"},
                            },
                        }],
                    },
                },
            },
        },
    },
}

# Load cluster config (in-cluster when this notebook runs on a RHOAI workbench).
try:
    config.load_incluster_config()
except config.ConfigException:
    config.load_kube_config()

api = client.CustomObjectsApi()
group, version, plural = "kubeflow.org", "v1", "pytorchjobs"

# Clean up any previous instance so we can re-run this cell freely.
try:
    api.delete_namespaced_custom_object(group, version, NAMESPACE, plural, JOB_NAME)
    print(f"Deleted previous PyTorchJob {JOB_NAME}")
    time.sleep(3)
except client.ApiException as e:
    if e.status != 404:
        raise

api.create_namespaced_custom_object(group, version, NAMESPACE, plural, pytorchjob)
print(f"Created PyTorchJob {NAMESPACE}/{JOB_NAME}. Watch with: oc logs -n {NAMESPACE} -f {JOB_NAME}-master-0")

## Wait for completion and stream the pod's logs

In [ ]:
def job_phase() -> str:
    obj = api.get_namespaced_custom_object(group, version, NAMESPACE, plural, JOB_NAME)
    conditions = (obj.get("status") or {}).get("conditions") or []
    # PyTorchJob conditions: Created -> Running -> Succeeded / Failed
    if conditions:
        latest = [c for c in conditions if c.get("status") == "True"]
        if latest:
            return latest[-1]["type"]
    return "Pending"


# Wait until the pod exists, then tail logs.
core = client.CoreV1Api()
pod_name = f"{JOB_NAME}-master-0"

deadline = time.time() + 60
while time.time() < deadline:
    try:
        core.read_namespaced_pod(pod_name, NAMESPACE)
        break
    except client.ApiException as e:
        if e.status != 404:
            raise
        print(f"  waiting for pod {pod_name} ... phase={job_phase()}")
        time.sleep(5)
else:
    raise TimeoutError(f"Pod {pod_name} never appeared within 60s")

print(f"--- streaming logs from {pod_name} ---")
# follow=True blocks until the container exits
for line in core.read_namespaced_pod_log(
    pod_name, NAMESPACE, follow=True, _preload_content=False
).stream():
    print(line.decode(), end="")

print(f"\n--- final PyTorchJob phase: {job_phase()} ---")

## Verify the outputs landed on S3

Same S3 contract as the Elyra pipeline, so [../05-register-and-deploy.ipynb](../05-register-and-deploy.ipynb) can proceed from here with `MODEL_URI` already pointing at this model version.

In [ ]:
import boto3
import botocore

_session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
_s3 = _session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
bucket = _s3.Bucket(os.environ["AWS_S3_BUCKET"])

s3_prefix = f"{MODEL_PREFIX}/{MODEL_VERSION}"
print(f"Contents of s3://{bucket.name}/{s3_prefix}/")
for obj in bucket.objects.filter(Prefix=f"{s3_prefix}/"):
    rel = obj.key[len(s3_prefix) + 1:]
    print(f"  {rel:25s}  {obj.size:>10} B")

---

**Next:** [../05-register-and-deploy.ipynb](../05-register-and-deploy.ipynb) — register this version in the Model Registry and roll out as a KServe InferenceService. The registration notebook reads from the same S3 prefix regardless of whether the training ran in the workbench, the Elyra pipeline, or this PyTorchJob.

**Clean up the job when you're done:**
```bash
oc delete pytorchjob -n ppe-detection-cv ppe-yolo-train
```
Or re-running the submit cell deletes and recreates automatically.